# Does recording quality matter for Bamboo speech biomarkers?

This notebook answers the "so what" question for Paper 1 with two small analyses of established
Bamboo-passage speech measures (Paper 2's six frozen primary acoustic features: articulation rate,
percent pause time, mean pause duration, mean phrase duration, phrase-duration CV, and F0 IQR).

| Part | Question | Data |
|---|---|---|
| B1 | When one acquisition factor is changed under control, does the matching quality measure move, do the other measures stay put, and how far do the biomarkers move relative to the real ALS–control difference? | Paper 2 Goal 3 controlled-perturbation outputs (same-source recordings, 6 transforms × 3 doses) |
| B2 | Do the real-cohort ALS–control and bulbar associations of those biomarkers change when recordings flagged by the structurally identifiable or support-separated quality measures (classes A and B) are excluded? | Frozen acoustic features joined to Notebook 02's analysis matrix |

**Prerequisites.** Notebook 02 must have completed a final run (its model helpers and analysis matrix are
rebuilt here exactly as in Notebook 08). The Paper 2 repository must be on the same computer with its
Goal 3 Stage E final outputs; point `PAPER2_ROOT` at it if it is not found automatically.

**Outputs.** `outputs/biomarker_impact/tables/*.csv`, `outputs/biomarker_impact/figures/` and
`outputs/biomarker_impact/biomarker_impact_summary.md`. All outputs are aggregate.

## 0. Configuration and inputs

In [ ]:
from __future__ import annotations

import json
import os
import platform
import warnings
from datetime import datetime, timezone
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from statsmodels.stats.multitest import multipletests
from IPython.display import display

SEED = 20261014
TASK = 'diagnosis'                 # Goal 3 source set containing both ALS and control recordings
SHIFT_DOSE = 'high'                # dose shown in the main figure; all doses are tabulated
FLAG_QUANTILE = 0.90               # primary: top decile of pause-region level / hum-comb
FLAG_QUANTILE_SENSITIVITY = 0.80   # sensitivity: top quintile
Q_THRESHOLD = 0.05

PRIMARY_A_LABELS = {
    'bamboo_nominal_articulation_rate_syll_per_sec': 'Articulation rate',
    'bamboo_percent_pause_time_300ms': 'Percent pause time',
    'bamboo_pause_mean_sec_300ms': 'Mean pause duration',
    'bamboo_phrase_mean_sec_300ms': 'Mean phrase duration',
    'bamboo_phrase_cv_300ms': 'Phrase-duration CV',
    'bamboo_f0_iqr_semitones': 'F0 IQR',
}
TRANSFORM_LABELS = {
    'stationary_colored_broadband': 'Additive noise',
    'uniform_level_shift': 'Level reduction',
    'smooth_time_varying_gain': 'Time-varying gain',
    'RIR_convolution_RMS_matched': 'Reverberation',
    'upper_band_restriction': 'Band limitation',
    'symmetric_hard_clipping': 'Hard clipping',
}
SEPARABILITY_CLASS = {
    'qadd_pause_ac_level_dbfs_median': 'B', 'qadd_pause_level_iqr_db': 'B',
    'qadd_speech_pause_level_contrast_db': 'C', 'qadd_pause_spectral_flatness': 'B',
    'qadd_mains_hum_comb_score_db': 'A',
    'qgain_typical_speech_level_dbfs': 'C', 'qgain_within_segment_iqr_db': 'C',
    'qgain_between_segment_mad_db': 'C', 'qgain_abs_drift_db_per_min': 'C',
    'qrev_tail_excess_100ms_db': 'C', 'qrev_tail_persistence_median_sec': 'C',
    'qrev_downward_decay_rate_db_per_sec': 'C', 'qrev_srmr_norm': 'C',
    'qchan_ltas_distance_db': 'C', 'qchan_rolloff95_deficit_hz': 'C',
    'qchan_highband_ratio_deficit': 'C', 'qchan_tilt_steepening_db_per_oct': 'C',
    'qdist_hard_clipped_sample_fraction': 'A', 'qdist_hard_clip_event_rate_per_min': 'A',
    'qdist_hard_clipped_frame_fraction': 'A',
}
QDIST_MEASURES = ['qdist_hard_clipped_sample_fraction', 'qdist_hard_clip_event_rate_per_min',
                  'qdist_hard_clipped_frame_fraction']
Q_SHORT = {
    'qadd_pause_ac_level_dbfs_median': 'Pause level', 'qadd_pause_level_iqr_db': 'Pause-level var.',
    'qadd_speech_pause_level_contrast_db': 'Speech–pause contrast',
    'qgain_typical_speech_level_dbfs': 'Speech level', 'qgain_within_segment_iqr_db': 'Within-seg. level IQR',
    'qgain_between_segment_mad_db': 'Between-seg. level MAD', 'qgain_abs_drift_db_per_min': 'Level drift',
    'qrev_srmr_norm': 'SRMR', 'qchan_ltas_distance_db': 'LTAS distance', 'qchan_rolloff95_deficit_hz': 'Roll-off deficit',
    'qchan_highband_ratio_deficit': 'High-band deficit', 'qchan_tilt_steepening_db_per_oct': 'Tilt steepening',
}
PAUSE_LEVEL = 'qadd_pause_ac_level_dbfs_median'
HUM = 'qadd_mains_hum_comb_score_db'


def locate_analysis_root() -> Path:
    configured = os.environ.get('PAPER1_ANALYSIS_ROOT', '').strip()
    if configured:
        return Path(configured).expanduser().resolve()
    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'requirements.txt').is_file() and (candidate / 'notebooks').is_dir():
            return candidate
    raise FileNotFoundError('Could not locate the analysis repository root')


def is_paper2_root(path: Path) -> bool:
    return (path / 'data' / 'processed' / 'acoustic_features_frozen.csv').is_file() and (
        path / 'outputs' / 'goal3').is_dir()


def locate_paper2_root(analysis_root: Path) -> Path:
    configured = os.environ.get('PAPER2_ROOT', '').strip()
    if configured:
        path = Path(configured).expanduser().resolve()
        if not is_paper2_root(path):
            raise FileNotFoundError(f'PAPER2_ROOT does not look like the Paper 2 repository: {path}')
        return path
    for parent in [analysis_root.parent, analysis_root.parent.parent]:
        for candidate in sorted(parent.glob('*')):
            if candidate.is_dir() and is_paper2_root(candidate):
                return candidate.resolve()
    raise FileNotFoundError(
        'Could not find the Paper 2 repository next to this one. Set PAPER2_ROOT to its folder '
        '(the one containing data\\processed\\acoustic_features_frozen.csv and outputs\\goal3).'
    )


ANALYSIS_ROOT = locate_analysis_root()
PAPER2_ROOT = locate_paper2_root(ANALYSIS_ROOT)
NB02_PATH = ANALYSIS_ROOT / 'notebooks' / '02_extent_distribution_persistence.ipynb'
GOAL3_TABLES = Path(os.environ.get('PAPER2_GOAL3_TABLES', '').strip() or (
    PAPER2_ROOT / 'outputs' / 'goal3' / 'stageE_controlled_perturbation_v1_0' / 'final' / 'tables'))
OUTPUT_ROOT = ANALYSIS_ROOT / 'outputs' / 'biomarker_impact'
TABLE_DIR = OUTPUT_ROOT / 'tables'
FIGURE_DIR = OUTPUT_ROOT / 'figures'
LOG_DIR = OUTPUT_ROOT / 'logs'
for directory in [TABLE_DIR, FIGURE_DIR, LOG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

REQUIRED_INPUTS = {
    'Paper 2 frozen acoustic values': PAPER2_ROOT / 'data' / 'processed' / 'acoustic_features_frozen.csv',
    'Paper 2 acoustic registry': PAPER2_ROOT / 'data' / 'manifests' / 'a_registry.csv',
    'Goal 3 feature deltas': GOAL3_TABLES / 'goal3_feature_delta.csv',
    'Goal 3 target verification': GOAL3_TABLES / 'goal3_target_q_verification.csv',
    'Goal 3 off-target Q summary': GOAL3_TABLES / 'goal3_offtarget_q_summary.csv',
}
missing = {name: str(path) for name, path in REQUIRED_INPUTS.items() if not path.is_file()}
if missing:
    raise FileNotFoundError('Missing inputs:\n' + '\n'.join(f'  {k}: {v}' for k, v in missing.items()))


def export(frame: pd.DataFrame, name: str) -> Path:
    path = TABLE_DIR / name
    frame.to_csv(path, index=False)
    return path


def bh(p_values: pd.Series) -> pd.Series:
    out = pd.Series(np.nan, index=p_values.index, dtype=float)
    mask = p_values.notna() & np.isfinite(p_values)
    if mask.any():
        out.loc[mask] = multipletests(p_values.loc[mask].astype(float), method='fdr_bh')[1]
    return out


print('ANALYSIS_ROOT:', ANALYSIS_ROOT)
print('PAPER2_ROOT:  ', PAPER2_ROOT)
print('GOAL3_TABLES: ', GOAL3_TABLES)

## 1. Rebuild the Notebook 02 analysis matrix and model helpers

Identical to Notebook 08: Notebook 02's input, harmonization, scaling, and model-helper cells are
re-executed in an isolated namespace, writing only under `outputs/revision_analyses/nb02_rebuild/`.

In [ ]:
NB02_SECTIONS = [
    '## 0. Frozen statistical specification',
    '## 1. Shared utilities',
    '## 2. Discover and hash immutable inputs',
    '## 3. Frozen feature registry',
    '## 4. Harmonize family tables',
    '## 5. Blocking population',
    '## 6. Availability, transforms',
    '## 8. Clustered-model helpers',
]
ORIGINAL_OUTPUT_LINE = "OUTPUT_ROOT = ANALYSIS_ROOT / 'outputs' / 'extent_distribution_persistence'"
REBUILD_OUTPUT_LINE = "OUTPUT_ROOT = ANALYSIS_ROOT / 'outputs' / 'revision_analyses' / 'nb02_rebuild'"
DISPLAY_IMPORT = 'from IPython.display import display, Markdown'

nb02 = json.loads(NB02_PATH.read_text(encoding='utf-8'))
nb02_cells = nb02['cells']


def code_after_heading(prefix: str) -> str:
    for index, cell in enumerate(nb02_cells):
        if cell['cell_type'] == 'markdown' and ''.join(cell['source']).lstrip().startswith(prefix):
            for following in nb02_cells[index + 1:]:
                if following['cell_type'] == 'code':
                    return ''.join(following['source'])
    raise KeyError(f'Notebook 02 section not found: {prefix!r}')


nb02_sources = [code_after_heading(prefix) for prefix in NB02_SECTIONS]
if ORIGINAL_OUTPUT_LINE not in nb02_sources[0] or DISPLAY_IMPORT not in nb02_sources[0]:
    raise AssertionError('Notebook 02 setup cell changed; update the rebuild substitutions.')
nb02_sources[0] = (
    nb02_sources[0]
    .replace(ORIGINAL_OUTPUT_LINE, REBUILD_OUTPUT_LINE)
    .replace(DISPLAY_IMPORT, '')
)

nb02_ns: dict = {'__name__': 'nb02_rebuild', 'display': lambda *args, **kwargs: None,
                 'Markdown': str}
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    for prefix, source in zip(NB02_SECTIONS, nb02_sources):
        exec(compile(source, f'<nb02 {prefix}>', 'exec'), nb02_ns)

analysis = nb02_ns['analysis']
FEATURE_REGISTRY = nb02_ns['FEATURE_REGISTRY']
fit_continuous_clustered = nb02_ns['fit_continuous_clustered']
fit_binary_gee = nb02_ns['fit_binary_gee']

assert len(analysis) == 519 and analysis['participant_id'].nunique() == 224
print('Rebuilt analysis matrix:', analysis.shape)

## 2. Join the frozen Bamboo acoustic features

Paper 2's frozen acoustic table shares the governed recording identifiers with Paper 1. The join must
be one-to-one over all 519 recordings, and participant identifiers must agree.

In [ ]:
a_registry = pd.read_csv(REQUIRED_INPUTS['Paper 2 acoustic registry'])
PRIMARY_A = a_registry.loc[a_registry['final_role'].eq('primary'), 'feature'].tolist()
if set(PRIMARY_A) != set(PRIMARY_A_LABELS):
    raise AssertionError(f'Unexpected primary acoustic features: {sorted(PRIMARY_A)}')
PRIMARY_A = list(PRIMARY_A_LABELS)          # fixed display order
A_TRANSFORM = a_registry.set_index('feature')['final_transform'].fillna('none').to_dict()

a_values = pd.read_csv(REQUIRED_INPUTS['Paper 2 frozen acoustic values'], low_memory=False)
a_values['logical_recording_id'] = a_values['logical_recording_id'].astype(str).str.strip()
a_values['participant_id'] = a_values['participant_id'].astype(str).str.strip()
if len(a_values) != 519 or not a_values['logical_recording_id'].is_unique:
    raise AssertionError('Paper 2 acoustic table must hold 519 unique recordings.')

keys = analysis[['logical_recording_id', 'participant_id']].astype(str)
check = keys.merge(a_values[['logical_recording_id', 'participant_id']],
                   on='logical_recording_id', how='outer', suffixes=('', '_p2'), indicator=True)
if not check['_merge'].eq('both').all():
    raise AssertionError('Recording identifiers differ between Paper 1 and Paper 2.')
if not (check['participant_id'] == check['participant_id_p2']).all():
    raise AssertionError('Participant identifiers differ between Paper 1 and Paper 2.')

data = analysis.copy()
data['logical_recording_id'] = data['logical_recording_id'].astype(str)
data = data.merge(a_values[['logical_recording_id'] + PRIMARY_A], on='logical_recording_id', how='left')

transform_feature = nb02_ns['transform_feature']
robust_scale = nb02_ns['robust_scale']
for feature in PRIMARY_A:
    transformed = transform_feature(data[feature], A_TRANSFORM.get(feature, 'none'))
    data[f'{feature}__available'] = transformed.notna()
    data[f'{feature}__z'], _, _ = robust_scale(transformed)   # full-cohort median / IQR, as in Notebook 02

availability = pd.DataFrame([{
    'feature': f, 'label': PRIMARY_A_LABELS[f], 'transform': A_TRANSFORM.get(f, 'none'),
    'available_recordings': int(data[f'{f}__available'].sum()),
    'available_als': int(data.loc[data['cohort'].eq('ALS'), f'{f}__available'].sum()),
    'available_control': int(data.loc[data['cohort'].eq('CONTROL'), f'{f}__available'].sum()),
} for f in PRIMARY_A])
export(availability, 'B0_acoustic_availability.csv')
display(availability)

## 3. Real ALS–control difference on the raw scale

The reference for "how much is a lot" is the observed ALS–control difference: the difference between
group medians of participant-mean values, in the feature's own unit. The natural IQR (all 519 recordings)
is reported alongside, as in Paper 2.

In [ ]:
participant_means = data.groupby(['participant_id', 'cohort'])[PRIMARY_A].mean().reset_index()
gap_rows = []
for feature in PRIMARY_A:
    als_values = participant_means.loc[participant_means['cohort'].eq('ALS'), feature].dropna()
    ctl_values = participant_means.loc[participant_means['cohort'].eq('CONTROL'), feature].dropna()
    raw = pd.to_numeric(data[feature], errors='coerce').dropna()
    gap_rows.append({
        'feature': feature, 'label': PRIMARY_A_LABELS[feature],
        'als_participants': len(als_values), 'control_participants': len(ctl_values),
        'als_median': als_values.median(), 'control_median': ctl_values.median(),
        'als_minus_control': als_values.median() - ctl_values.median(),
        'natural_iqr_raw': raw.quantile(0.75) - raw.quantile(0.25),
    })
gap = pd.DataFrame(gap_rows)
gap['gap_in_natural_iqr'] = gap['als_minus_control'] / gap['natural_iqr_raw']
export(gap, 'B1_als_control_gap_raw.csv')
display(gap)
GAP = gap.set_index('feature')['als_minus_control'].to_dict()
A_IQR = gap.set_index('feature')['natural_iqr_raw'].to_dict()

## B1. Controlled degradation: specificity of the quality measures and movement of the biomarkers

Read from Paper 2 Goal 3 Stage E (diagnosis source set: one recording per participant, each degraded at
three calibrated doses, segmentation rerun after each degradation). Nothing is recomputed from audio.

* **B1a** carries over the target dose response.
* **B1b** is the specificity matrix: the median absolute change of every core quality measure, in natural-IQR
  units, under each degradation. Hard clipping is checked separately as a false-positive rate, because
  the clipping measure is zero in nearly every natural recording.
* **B1c** expresses each biomarker's shift in its own unit, in natural-IQR units, and as a percentage of
  the real ALS–control difference (positive = the recording moves toward the ALS side).

In [ ]:
target_q = pd.read_csv(REQUIRED_INPUTS['Goal 3 target verification'])
target_q = target_q.loc[target_q['task'].eq(TASK)].copy()
target_q['transform_label'] = target_q['transform'].map(TRANSFORM_LABELS)
export(target_q, 'B1a_target_q_dose_response.csv')

offtarget = pd.read_csv(REQUIRED_INPUTS['Goal 3 off-target Q summary'])
offtarget = offtarget.loc[offtarget['task'].eq(TASK)].copy()
offtarget['q_family'] = offtarget['q_feature'].str.split('_').str[0].str.upper()
offtarget['separability_class'] = offtarget['q_feature'].map(SEPARABILITY_CLASS)
offtarget['transform_label'] = offtarget['transform'].map(TRANSFORM_LABELS)
export(offtarget, 'B1b_specificity_matrix_long.csv')

spec_rows = []
for (transform, dose), g in offtarget.groupby(['transform', 'dose_label']):
    target = g.loc[g['is_primary_target']]
    other_family = g.loc[~g['is_primary_target'] & g['q_family'].ne(g['family'].str.upper())]
    worst = other_family.sort_values('median_abs_delta_natural_iqr', ascending=False).head(1)
    spec_rows.append({
        'transform': transform, 'transform_label': TRANSFORM_LABELS.get(transform, transform),
        'dose_label': dose, 'target_family': g['family'].iloc[0],
        'target_q': target['q_feature'].iloc[0] if len(target) else None,
        'target_median_abs_iqr': target['median_abs_delta_natural_iqr'].iloc[0] if len(target) else np.nan,
        'max_other_family_q': worst['q_feature'].iloc[0] if len(worst) else None,
        'max_other_family_median_abs_iqr': worst['median_abs_delta_natural_iqr'].iloc[0] if len(worst) else np.nan,
        'median_other_family_abs_iqr': other_family['median_abs_delta_natural_iqr'].median(),
    })
specificity = pd.DataFrame(spec_rows)
specificity = specificity.sort_values(
    ['transform', 'dose_label'],
    key=lambda c: c.map({t: k for k, t in enumerate(TRANSFORM_LABELS)}) if c.name == 'transform'
    else c.map({'low': 1, 'medium': 2, 'high': 3}))
specificity.loc[specificity['transform'].eq('symmetric_hard_clipping'), 'target_q'] = (
    'qdist_hard_clipped_sample_fraction (see clipping detection)')
specificity['target_to_max_other_ratio'] = (
    specificity['target_median_abs_iqr'] / specificity['max_other_family_median_abs_iqr'])
export(specificity, 'B1b_specificity_summary.csv')

deltas = pd.read_csv(REQUIRED_INPUTS['Goal 3 feature deltas'], low_memory=False)
deltas = deltas.loc[deltas['task'].eq(TASK)].copy()
deltas['delta'] = pd.to_numeric(deltas['delta'], errors='coerce')

qdist = deltas.loc[deltas['feature'].eq('qdist_hard_clipped_sample_fraction')].copy()
qdist['baseline_zero'] = pd.to_numeric(qdist['baseline_value'], errors='coerce').eq(0)
qdist['perturbed_positive'] = pd.to_numeric(qdist['perturbed_value'], errors='coerce').gt(0)
clip_fp = (
    qdist.loc[qdist['baseline_zero']]
    .groupby(['transform', 'dose_label'], as_index=False)
    .agg(sources=('participant_id', 'nunique'), clipping_detected_fraction=('perturbed_positive', 'mean'))
)
clip_fp['transform_label'] = clip_fp['transform'].map(TRANSFORM_LABELS)
DOSE_ORDER = {'low': 1, 'medium': 2, 'high': 3}
TRANSFORM_ORDER = {t: k for k, t in enumerate(TRANSFORM_LABELS)}
clip_fp = clip_fp.sort_values(['transform', 'dose_label'],
                              key=lambda c: c.map(TRANSFORM_ORDER) if c.name == 'transform' else c.map(DOSE_ORDER))
export(clip_fp, 'B1b_clipping_detection_by_transform.csv')

a_delta = deltas.loc[deltas['measurement_type'].eq('A') & deltas['feature'].isin(PRIMARY_A)].copy()
a_delta['gap'] = a_delta['feature'].map(GAP)
a_delta['iqr'] = a_delta['feature'].map(A_IQR)
a_delta['delta_iqr'] = a_delta['delta'] / a_delta['iqr']
a_delta['toward_als'] = a_delta['delta'] * np.sign(a_delta['gap'])
a_delta['support_lost'] = (
    pd.to_numeric(a_delta['baseline_supported'], errors='coerce').eq(1)
    & pd.to_numeric(a_delta['perturbed_support_fraction'], errors='coerce').eq(0)
)


def shift_summary(g: pd.DataFrame) -> pd.Series:
    d = g['delta'].dropna()
    gap_value = g['gap'].iloc[0]
    rel = d / abs(gap_value) if np.isfinite(gap_value) and gap_value != 0 else d * np.nan
    return pd.Series({
        'sources': g['participant_id'].nunique(),
        'finite_deltas': len(d),
        'support_lost_fraction': g['support_lost'].mean(),
        'median_delta_raw': d.median(),
        'q25_delta_raw': d.quantile(0.25),
        'q75_delta_raw': d.quantile(0.75),
        'median_delta_natural_iqr': (d / g['iqr'].iloc[0]).median(),
        'median_shift_pct_of_gap_toward_als': 100 * (g['toward_als'].dropna() / abs(gap_value)).median()
        if np.isfinite(gap_value) and gap_value != 0 else np.nan,
        'fraction_moving_toward_als': (g['toward_als'].dropna() > 0).mean(),
        'fraction_abs_shift_ge_half_gap': (rel.abs() >= 0.5).mean(),
    })


with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    shifts = (a_delta.groupby(['transform', 'dose_label', 'dose_code', 'feature'])
              .apply(shift_summary, include_groups=False).reset_index())
shifts['transform_label'] = shifts['transform'].map(TRANSFORM_LABELS)
shifts['label'] = shifts['feature'].map(PRIMARY_A_LABELS)
shifts = shifts.sort_values(['transform', 'dose_code', 'feature'])
export(shifts, 'B1c_biomarker_shift_vs_gap.csv')

display(specificity.loc[specificity['dose_label'].eq(SHIFT_DOSE)])
display(clip_fp)
display(shifts.loc[shifts['dose_label'].eq(SHIFT_DOSE),
                   ['transform_label', 'label', 'median_delta_raw', 'median_delta_natural_iqr',
                    'median_shift_pct_of_gap_toward_als', 'fraction_abs_shift_ge_half_gap',
                    'support_lost_fraction']])

## B2. Real-cohort biomarker associations with and without technically flagged recordings

Only quality measures that speech production cannot generate or that are measured where target speech is
absent (classes A and B) are used as flags, because they can be read as acquisition evidence:

* any hard clipping (QDIST, class A);
* mains-hum comb score in the cohort's top decile (class A, where available);
* pause-region level in the cohort's top decile (class B: the noisiest recording conditions).

The primary-A associations are refitted with Notebook 02's model routes (Gaussian mixed model with a
participant random intercept, GEE fallback) for ALS vs control (unadjusted, and age- and sex-adjusted on
the same complete cases) and for ALSFRS-R bulbar score (between and within participant, with Notebook 02's
gates). Variants: all recordings; flagged recordings excluded (top decile, primary; top quintile,
sensitivity); and adjustment for pause-region level as a covariate, beside an unadjusted model on the same
recordings. Scaling is fixed to the full cohort so estimates are comparable across variants. BH correction
is applied across the six features within each variant and term.

In [ ]:
def top_flag(feature: str, quantile: float) -> pd.Series:
    values = pd.to_numeric(data[feature], errors='coerce').where(data[f'{feature}__available'])
    cut = values.quantile(quantile)
    return values.gt(cut).fillna(False)


clip_any = np.zeros(len(data), dtype=bool)
for f in QDIST_MEASURES:
    clip_any |= data[f'{f}__positive'].fillna(False).to_numpy(bool)
data['flag_clipping'] = clip_any
for label, q in [('p90', FLAG_QUANTILE), ('p80', FLAG_QUANTILE_SENSITIVITY)]:
    data[f'flag_hum_{label}'] = top_flag(HUM, q)
    data[f'flag_pause_level_{label}'] = top_flag(PAUSE_LEVEL, q)
    data[f'flag_any_{label}'] = data['flag_clipping'] | data[f'flag_hum_{label}'] | data[f'flag_pause_level_{label}']

flag_rows = []
for flag in ['flag_clipping', 'flag_hum_p90', 'flag_pause_level_p90', 'flag_any_p90',
             'flag_hum_p80', 'flag_pause_level_p80', 'flag_any_p80']:
    row = {'flag': flag}
    for cohort in ['ALS', 'CONTROL']:
        sub = data.loc[data['cohort'].eq(cohort)]
        row[f'{cohort.lower()}_recordings_flagged'] = int(sub[flag].sum())
        row[f'{cohort.lower()}_recordings'] = len(sub)
        row[f'{cohort.lower()}_pct_flagged'] = 100 * sub[flag].mean()
    d = data.assign(flagged=data[flag].astype(int))
    if d['flagged'].nunique() == 2:
        result, _ = fit_binary_gee(d, 'flagged', 'cohort_als', feature=flag,
                                   analysis_name='flag_by_cohort', target_terms=['cohort_als'])
        if result:
            row.update({'cohort_log_odds': result[0]['estimate'], 'cohort_p_value': result[0]['p_value']})
    flag_rows.append(row)
flag_table = pd.DataFrame(flag_rows)
export(flag_table, 'B2a_flag_rates_by_cohort.csv')
display(flag_table)

MIN_BULBAR_MATCHED_RECORDINGS = nb02_ns['MIN_BULBAR_MATCHED_RECORDINGS']
MIN_BULBAR_REPEATED_PARTICIPANTS = nb02_ns['MIN_BULBAR_REPEATED_PARTICIPANTS']

data['age_centered'] = data['age_years'] - data['age_years'].mean(skipna=True)
data['sex_female'] = data['sex'].map({'MALE': 0.0, 'FEMALE': 1.0})
data['pause_level_z'] = data[f'{PAUSE_LEVEL}__z']


def fit_variant(frame: pd.DataFrame, variant: str, extra: str = '') -> list[dict]:
    rows = []
    for feature in PRIMARY_A:
        d = frame.loc[frame[f'{feature}__available']].copy()
        outcome = f'{feature}__z'
        r, _ = fit_continuous_clustered(d, outcome, 'cohort_als' + extra, feature=feature,
                                        analysis_name='cohort', target_terms=['cohort_als'])
        rows += [dict(x, variant=variant) for x in r]
        if not extra:
            dc = d.dropna(subset=[outcome, 'age_centered', 'sex_female'])
            r, _ = fit_continuous_clustered(dc, outcome, 'cohort_als + age_centered + sex_female',
                                            feature=feature, analysis_name='cohort_age_sex_adjusted',
                                            target_terms=['cohort_als'])
            rows += [dict(x, variant=variant) for x in r]
        b = d.loc[d['cohort'].eq('ALS') & d['bulbar_score_matched'].notna()].copy()
        if b.empty:
            continue
        mean = b.groupby('participant_id')['bulbar_score_matched'].transform('mean')
        b['bulbar_between'] = mean - b['bulbar_score_matched'].mean()
        b['bulbar_within'] = b['bulbar_score_matched'] - mean
        counts = b.groupby('participant_id')['bulbar_score_matched'].count()
        distinct = b.groupby('participant_id')['bulbar_score_matched'].nunique()
        if len(b) < MIN_BULBAR_MATCHED_RECORDINGS or b['participant_id'].nunique() < 30:
            continue
        within = (counts.ge(2).sum() >= MIN_BULBAR_REPEATED_PARTICIPANTS
                  and distinct.ge(2).sum() >= MIN_BULBAR_REPEATED_PARTICIPANTS)
        rhs = 'bulbar_between + bulbar_within' if within else 'bulbar_between'
        targets = ['bulbar_between', 'bulbar_within'] if within else ['bulbar_between']
        r, _ = fit_continuous_clustered(b, outcome, rhs + extra, feature=feature,
                                        analysis_name='bulbar', target_terms=targets)
        rows += [dict(x, variant=variant) for x in r]
    return rows


pause_known = data[f'{PAUSE_LEVEL}__available']
VARIANTS = [
    ('all_recordings', data, ''),
    ('exclude_flagged_p90', data.loc[~data['flag_any_p90']], ''),
    ('exclude_flagged_p80', data.loc[~data['flag_any_p80']], ''),
    ('pause_level_known_unadjusted', data.loc[pause_known], ''),
    ('pause_level_known_adjusted', data.loc[pause_known], ' + pause_level_z'),
]
model_rows = []
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    for name, frame, extra in VARIANTS:
        model_rows += fit_variant(frame, name, extra)
models = pd.DataFrame(model_rows)
models['label'] = models['feature'].map(PRIMARY_A_LABELS)
models['term'] = models['analysis'] + ':' + models['target_term']
models['q_value'] = models.groupby(['variant', 'term'])['p_value'].transform(bh)
models['significant'] = models['q_value'].lt(Q_THRESHOLD)
export(models, 'B2b_biomarker_models_by_variant.csv')

COMPARATOR = {
    'exclude_flagged_p90': 'all_recordings',
    'exclude_flagged_p80': 'all_recordings',
    'pause_level_known_adjusted': 'pause_level_known_unadjusted',
}
reference_cols = ['feature', 'term', 'estimate', 'q_value', 'significant', 'n_recordings']
comparison = pd.concat([
    models.loc[models['variant'].eq(variant)].merge(
        models.loc[models['variant'].eq(ref), reference_cols], on=['feature', 'term'], suffixes=('', '_ref'),
    ).assign(reference=ref)
    for variant, ref in COMPARATOR.items()
], ignore_index=True)
comparison['estimate_change'] = comparison['estimate'] - comparison['estimate_ref']
comparison['estimate_change_pct'] = 100 * comparison['estimate_change'] / comparison['estimate_ref'].abs()
comparison['conclusion_changed'] = comparison['significant'] != comparison['significant_ref']
comparison = comparison[['variant', 'reference', 'feature', 'label', 'term', 'n_recordings_ref', 'n_recordings',
                         'estimate_ref', 'estimate', 'estimate_change', 'estimate_change_pct',
                         'q_value_ref', 'q_value', 'significant_ref', 'significant', 'conclusion_changed']]
export(comparison, 'B2c_change_after_flag_handling.csv')
display(comparison.loc[comparison['variant'].eq('exclude_flagged_p90')])

## 4. Figure (candidate Paper 1 Fig. 5)

In [ ]:
matplotlib.rcParams.update({'font.size': 8, 'axes.titlesize': 9})
fig = plt.figure(figsize=(7.16, 6.4))
grid = fig.add_gridspec(2, 2, height_ratios=[1, 1.05], hspace=0.55, wspace=0.65)

# (a) Specificity of the quality measures at the shown dose.
ax = fig.add_subplot(grid[0, 0])
order_t = [t for t in TRANSFORM_LABELS if t in set(offtarget['transform']) and t != 'symmetric_hard_clipping']
q_order = [q for q in Q_SHORT if q in set(offtarget['q_feature'])]
heat = (offtarget.loc[offtarget['dose_label'].eq(SHIFT_DOSE)]
        .pivot_table(index='transform', columns='q_feature', values='median_abs_delta_natural_iqr')
        .reindex(index=order_t, columns=q_order))
im = ax.imshow(heat.to_numpy(float), cmap='Greys', vmin=0, vmax=max(1.0, np.nanmax(heat.to_numpy(float))),
               aspect='auto')
for i, t in enumerate(order_t):
    j = q_order.index(offtarget.loc[(offtarget['transform'] == t) & offtarget['is_primary_target'],
                                    'q_feature'].iloc[0])
    ax.add_patch(plt.Rectangle((j - .5, i - .5), 1, 1, fill=False, ec='#c0392b', lw=1.4))
ax.set_yticks(range(len(order_t)), [TRANSFORM_LABELS[t] for t in order_t])
ax.set_xticks(range(len(q_order)), [Q_SHORT[q] for q in q_order], rotation=60, ha='right', fontsize=6)
ax.set_title('a  Quality-measure response (|Δ|, natural IQR)', loc='left')
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03)

# (b) Biomarker shifts as % of the real ALS-control difference.
ax = fig.add_subplot(grid[0, 1])
order_all = [t for t in TRANSFORM_LABELS if t in set(shifts['transform'])]
pct = (shifts.loc[shifts['dose_label'].eq(SHIFT_DOSE)]
       .pivot_table(index='transform', columns='feature', values='median_shift_pct_of_gap_toward_als')
       .reindex(index=order_all, columns=PRIMARY_A))
limit = 100
im = ax.imshow(np.clip(pct.to_numpy(float), -limit, limit), cmap='RdBu_r', vmin=-limit, vmax=limit, aspect='auto')
for i in range(pct.shape[0]):
    for j in range(pct.shape[1]):
        v = pct.iloc[i, j]
        if np.isfinite(v):
            ax.text(j, i, f'{v:.0f}', ha='center', va='center', fontsize=6,
                    color='white' if abs(v) > 60 else 'black')
ax.set_yticks(range(len(order_all)), [TRANSFORM_LABELS[t] for t in order_all])
ax.set_xticks(range(len(PRIMARY_A)), [PRIMARY_A_LABELS[f] for f in PRIMARY_A], rotation=45, ha='right')
ax.set_title('b  Biomarker shift (% of ALS–control gap)', loc='left')
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03)

# (c) Real-cohort ALS-control estimates with and without flagged recordings.
ax = fig.add_subplot(grid[1, :])
show = models.loc[models['term'].eq('cohort:cohort_als')
                  & models['variant'].isin(['all_recordings', 'exclude_flagged_p90'])]
for k, (variant, style) in enumerate([('all_recordings', dict(color='black', marker='o', label='All recordings')),
                                      ('exclude_flagged_p90', dict(color='#c0392b', marker='s',
                                                                   label='Flagged recordings excluded'))]):
    s = show.loc[show['variant'].eq(variant)].set_index('feature').reindex(PRIMARY_A)
    y = np.arange(len(PRIMARY_A)) + (k - 0.5) * 0.25
    ax.errorbar(s['estimate'], y, xerr=[s['estimate'] - s['ci95_low'], s['ci95_high'] - s['estimate']],
                fmt=style['marker'], color=style['color'], ms=4, capsize=2, lw=1, label=style['label'])
ax.axvline(0, color='grey', lw=0.6)
ax.set_yticks(range(len(PRIMARY_A)), [PRIMARY_A_LABELS[f] for f in PRIMARY_A])
ax.invert_yaxis()
ax.set_xlabel('ALS − control (full-cohort IQR units)')
ax.set_title('c  ALS–control difference in the real cohort', loc='left')
ax.legend(frameon=False, loc='lower right')

for ext in ['png', 'pdf']:
    fig.savefig(FIGURE_DIR / f'fig5_biomarker_impact.{ext}', dpi=300, bbox_inches='tight')
plt.show()

## 5. Summary for the manuscript

In [ ]:
def fmt(x, digits=2):
    return 'NA' if x is None or not np.isfinite(x) else f'{x:.{digits}f}'


lines = ['# Biomarker impact summary', '',
         f'Generated {datetime.now(timezone.utc).isoformat(timespec="seconds")}. '
         f'Goal 3 task: {TASK}; dose shown: {SHIFT_DOSE}.', '']

lines += ['## Real ALS–control difference (raw units)', '',
          '| Biomarker | ALS median | Control median | ALS − control | in natural IQR |', '|---|---|---|---|---|']
for r in gap.itertuples():
    lines.append(f'| {r.label} | {fmt(r.als_median, 3)} | {fmt(r.control_median, 3)} | '
                 f'{fmt(r.als_minus_control, 3)} | {fmt(r.gap_in_natural_iqr)} |')

lines += ['', f'## B1b Specificity of quality measures ({SHIFT_DOSE} dose)', '',
          '| Degradation | Target measure | Target |Δ| (IQR) | Largest other-family |Δ| (IQR) | Ratio |',
          '|---|---|---|---|---|']
for r in specificity.loc[specificity['dose_label'].eq(SHIFT_DOSE)].itertuples():
    lines.append(f'| {r.transform_label} | {r.target_q} | {fmt(r.target_median_abs_iqr)} | '
                 f'{fmt(r.max_other_family_median_abs_iqr)} ({r.max_other_family_q}) | '
                 f'{fmt(r.target_to_max_other_ratio, 1)} |')
lines += ['', 'Clipping detected in recordings with no natural clipping, by degradation and dose:', '']
for r in clip_fp.itertuples():
    lines.append(f'- {r.transform_label}, {r.dose_label}: {100 * r.clipping_detected_fraction:.1f}% of {r.sources}')

lines += ['', f'## B1c Biomarker shift under degradation ({SHIFT_DOSE} dose)', '',
          'Median shift toward ALS as % of the real ALS–control difference '
          '(share of recordings shifted by at least half the difference).', '',
          '| Degradation | ' + ' | '.join(PRIMARY_A_LABELS[f] for f in PRIMARY_A) + ' |',
          '|---' * (len(PRIMARY_A) + 1) + '|']
for t in [t for t in TRANSFORM_LABELS if t in set(shifts['transform'])]:
    s = shifts.loc[shifts['transform'].eq(t) & shifts['dose_label'].eq(SHIFT_DOSE)].set_index('feature')
    cells_ = [f"{fmt(s.loc[f, 'median_shift_pct_of_gap_toward_als'], 0)}% "
              f"({fmt(100 * s.loc[f, 'fraction_abs_shift_ge_half_gap'], 0)}%)" if f in s.index else 'NA'
              for f in PRIMARY_A]
    lines.append(f'| {TRANSFORM_LABELS[t]} | ' + ' | '.join(cells_) + ' |')

lines += ['', '## B2a Technical flags by cohort', '',
          '| Flag | ALS % | Control % | cohort log-odds | p |', '|---|---|---|---|---|']
for r in flag_table.itertuples():
    lines.append(f'| {r.flag} | {fmt(r.als_pct_flagged, 1)} | {fmt(r.control_pct_flagged, 1)} | '
                 f'{fmt(getattr(r, "cohort_log_odds", np.nan))} | {fmt(getattr(r, "cohort_p_value", np.nan), 3)} |')

lines += ['', '## B2c Real-cohort associations with flagged recordings handled', '',
          '| Variant (vs reference) | Biomarker | Term | n | Estimate | q | Conclusion changed |',
          '|---|---|---|---|---|---|---|']
for r in comparison.itertuples():
    lines.append(f'| {r.variant} (vs {r.reference}) | {r.label} | {r.term} | {r.n_recordings_ref:.0f} → {r.n_recordings:.0f} | '
                 f'{fmt(r.estimate_ref)} → {fmt(r.estimate)} | {fmt(r.q_value_ref, 3)} → {fmt(r.q_value, 3)} | '
                 f'{"YES" if r.conclusion_changed else "no"} |')
n_changed = int(comparison.loc[comparison['variant'].eq('exclude_flagged_p90'), 'conclusion_changed'].sum())
lines += ['', f'Conclusions changed after excluding flagged recordings (top decile): {n_changed}.']

summary_path = OUTPUT_ROOT / 'biomarker_impact_summary.md'
summary_path.write_text('\n'.join(lines) + '\n', encoding='utf-8')

manifest = {
    'generated_utc': datetime.now(timezone.utc).isoformat(timespec='seconds'),
    'python': platform.python_version(), 'pandas': pd.__version__,
    'analysis_root': str(ANALYSIS_ROOT), 'paper2_root': str(PAPER2_ROOT), 'goal3_tables': str(GOAL3_TABLES),
    'task': TASK, 'shift_dose': SHIFT_DOSE, 'flag_quantile': FLAG_QUANTILE,
    'flag_quantile_sensitivity': FLAG_QUANTILE_SENSITIVITY, 'primary_a': PRIMARY_A,
    'outputs': sorted(p.name for p in TABLE_DIR.glob('*.csv')),
}
(LOG_DIR / 'run_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print(summary_path.read_text(encoding='utf-8'))